
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 4 - Building Dynamic Workloads with Advanced Tasks

In this demo, we will show how to build dynamic Lakeflow jobs using conditional logic (`if-else`) and iterative tasks (`for each` loop).

This demo will cover:
- Defining dependencies between tasks
- Adding a conditional `if-else` task
- Adding an iterative `for each` task

### Learning Objective
Create and visualize a dynamic Lakeflow job with multiple tasks and dependencies.

![Lesson04_final_job](./Includes/images/Lesson04_final_job.png)

After completing this demo, your job will look like above.

## REQUIRED - SELECT CLASSIC COMPUTE (The cluster named 'labuser')

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:


1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.

   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.
<br></br>


```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The `DA` object is only used in Databricks Academy courses and is not available outside of these courses. It will dynamically reference the information needed to run the course.

**NOTE:** If you use Serverless V1 a warning will be returned. You can ignore the warning.

In [0]:
%run ./Includes/Classroom-Setup-4

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


Course Catalog:,
Your Schema:,


Data has been copied from /Volumes/dbacademy_retail/v01/source_files/customers.csv to /Volumes/dbacademy/labuser15105141_1778791941/trigger_storage_location/


## B. Explore Your Schema
Complete the following to explore your **dbacademy.labuser** schema:

1. In the left navigation bar, select the catalog icon:  ![Catalog Icon](./Includes/images/catalog_icon.png)

2. Locate the catalog called **dbacademy** and expand the catalog.

3. Expand your **labuser** schema. 

4. Notice that within your schema you will find two tables named as **sales_bronze**, **customers_bronze** and **orders_bronze**.

**Note:** If you have completed the 2L Exercise, you may find additional tables under your schema.


## C. View Your Notebook

Follow these steps to view the notebook files used in this job. All files are located in the **Task Files** folder within the directory for the corresponding lesson number.

1. Navigate to (or click the link for) the notebook: [Task Files/Lesson 4 Files/4.1 - Joining Customers and Sales Table]($./Task Files/Lesson 4 Files/4.1 - Joining Customers and Sales Table)
   - Review the notebook. It creates a new table by joining the **customers_bronze** and **sales_bronze** tables. Pay attention to the code that sets the task value for the key **has_duplicates**.

2. Navigate to (or click the link for) the notebook: [Task Files/Lesson 4 Files/4.2 - Joining Customers and Orders Table]($./Task Files/Lesson 4 Files/4.2 - Joining Customers and Orders Table)
   - Review the notebook. It creates a new table by joining the **customers_bronze** and **orders_bronze** tables.

## D. Adding Task in Job
Complete the steps below to add new task into your Retail Job

###D1. Creating the Starter Job
1. Next, we will add the notebooks listed below as tasks to our job using the Databricks SDK. This approach avoids manually adding notebook tasks, as we've already done it in previous demonstrations and labs:


   - **4.1 - Joining Customers and Sales Table**  

   - **4.2 - Joining Customers and Orders Table**  

   Run the cell below to build the starter job that we have been continually building in this course. These commands will set up your job with all work completed so far and add the required tasks for this demonstration.

![Lesson04_starter_job](./Includes/images/Lesson04_starter_job.png)



In [0]:
job_tasks = [
        {
            'task_name': 'ingesting_orders',
            'file_path': '/Task Files/Lesson 1 Files/1.1 - Creating orders table',
            'depends_on': None
        },
        {
            'task_name': 'ingesting_sales',
            'file_path': '/Task Files/Lesson 1 Files/1.2 - Creating sales table',
            'depends_on': None
        },
        {
            'task_name': 'ingesting_customers',
            'file_path': '/Task Files/Lesson 3 Files/3.1 - Creating customers table',
            'depends_on': None
        }
        ,{
            'task_name': 'customers_sales_summary',
            'file_path': '/Task Files/Lesson 4 Files/4.1 - Joining Customers and Sales Table',
            'depends_on': [
                        {'task_key':'ingesting_customers'},
                        {'task_key': 'ingesting_sales'}
                        ]
        }
        ,{
            'task_name' : 'customers_orders_report',
            'file_path': '/Task Files/Lesson 4 Files/4.2 - Joining Customers and Orders Table',
            'depends_on': None
        }
    ]

myjob = DAJobConfig(job_name=f"Demo_04_Retail_Job_{DA.schema_name}",
                    job_tasks=job_tasks,
                    job_parameters=[
                        {'name':'catalog', 'default':'dbacademy'},
                        {'name':'schema', 'default':f'{DA.schema_name}'}
                    ])

Job name is unique. Creating the job Demo_04_Retail_Job_labuser15105141_1778791941...
Using the following path to reference the Files: /Workspace/Users/labuser15105141_1778791941@vocareum.com/deploy-workloads-with-lakeflow-jobs-en_us-3.2.4/Deploy Workloads with Lakeflow Jobs/.


### D2. Set Dependencies on the Tasks

In this step, we will modify the existing job to define task dependencies. Specifically, we'll configure the main task to run only after all preceding tasks have completed successfully.


Complete the following to review the job and set the following dependencies to the **customers_orders_report** task:
   - **ingesting_orders**
   - **ingesting_customers**

1. Navigate to **Jobs and Pipelines** and open it in a new tab.

2. Select your new job that starts with **Demo_04_Retail_Job_labuser**.

3. Click on **Tasks** in the top navigation bar.

4. Review your job. You should see five tasks: 
   - **customers_orders_report**.
   - **ingesting_customers**, 
   - **ingesting_orders**, 
   - **ingesting_sales**, 
   - **customers_sales_summary**,

5. Select the **customers_sales_summary** task. 
   - Notice that it depends on two tasks: **ingesting_customers** and **ingesting_sales**, with the dependency set to **All Succeeded**.

6. Next, select the **customers_orders_report** task and set the following task options: 

   - In the **Depends on**, add **ingesting_orders** and **ingesting_customers**

   - In **Run if dependencies**, set the dependency to **All Succeeded**.

   - Select **Save task**.

7. Click on **Run_now** to run the job.

<br></br>
#### Final Dependencies
![Lesson04_dependencies](./Includes/images/Lesson04_dependencies.png)


## E. Adicione uma Tarefa Condicional If/Else

Nesta seção, você irá adicionar uma tarefa condicional ao seu job que verifica registros duplicados na tabela **customers_sales_silver** (Tarefa **customers_sales_summary**).

Com base no resultado, o fluxo de trabalho irá ramificar para lidar com duplicatas de forma apropriada.

### E1. Lógica de Verificação de Duplicatas

1. Relembre a lógica usada para detectar duplicatas na tabela **customers_sales_silver**. (A tarefa **customers_sales_summary** cria a tabela **customers_sales_silver**.)

2. Nesse notebook, verificamos se a tabela **customers_sales_silver** contém registros duplicados. Se duplicatas forem encontradas, o resultado dessa verificação (um valor booleano) é armazenado como `has_duplicates` na saída da tarefa.

**Referência de Código:**

        df = spark.sql("""
            SELECT * FROM customers_sales_silver
        """)

        duplicate_exists = df.count() > df.dropDuplicates().count()

        dbutils.jobs.taskValues.set(key="has_duplicates", value=duplicate_exists)

**Notebook para Referência:**  
[Task Files/Lesson 4 Files/4.1 - Joining Customers and Sales Table]($./Task Files/Lesson 4 Files/4.1 - Joining Customers and Sales Table)

### E2. Criar uma Tarefa Condicional If/Else

Crie uma tarefa condicional **If/else** para determinar o que executar com base na existência de registros duplicados.

1. No seu job **Demo_04_Retail_Job_labuser**, selecione **Adicionar tarefa**.

2. Na caixa de diálogo, role até a seção **Avançado** e selecione o tipo de tarefa **Condição If/else**.

3. Nomeie a nova tarefa como **checking_for_duplicates**.

4. Defina o valor **Depende de** para a tarefa **customers_sales_summary**.

5. Para o campo **Condição**, use o valor do parâmetro criado na tarefa `customers_sales_summary`:

   **Referências de Valor Dinâmico:**
   Essa sintaxe utiliza referências dinâmicas para acessar variáveis de saída de tarefas anteriores no seu job. Quando uma tarefa é executada (como `customers_sales_summary`), seus resultados — incluindo variáveis registradas ou de saída (como `has_duplicates`) — ficam disponíveis para tarefas posteriores.

   **Ao referenciar** `tasks.customers_sales_summary.values.has_duplicates`, você passa dinamicamente o valor (se existem duplicatas) para a condição If/Else. Isso permite ramificação condicional baseada em dados de tempo de execução, tornando seu fluxo de trabalho adaptável e responsivo aos resultados reais.

    **Adicionando o Campo de Condição:** 
     - Para adicionar manualmente o valor do parâmetro, selecione o `{}` no campo **Condição**. 
     - Encontre e clique em `tasks.customers_sales_summary.values`, ele adicionará automaticamente o sufixo `my_value`.
     - Substitua `my_value` pelo parâmetro criado no notebook: `has_duplicates`.

6. Em seguida, defina a condição para verificar se esse valor  `== true`

7. Selecione **Salvar tarefa** para criar a tarefa condicional.


<br></br>
##### TAREFA DE CONDIÇÃO IF/ELSE

![Lesson04_conditional_task.png](./Includes/images/Lesson04_conditional_task.png)

### E3. Defina a Tarefa para Condição Verdadeira (Duplicatas Existem)

Complete os passos abaixo para adicionar uma tarefa que será executada **apenas se forem encontradas duplicatas** (`tasks.customers_sales_summary.values.has_duplicates == true`).

1. Selecione a tarefa **checking_for_duplicates**.

2. Clique em **Adicionar tarefa** e escolha **Notebook**.

3. Nomeie a nova tarefa como **dropping_duplicate_records**.

4. Use o notebook [4.3 - If Condition: Dropping Duplicates]($./Task Files/Lesson 4 Files/4.3 - If Condition: Dropping Duplicates) como fonte da tarefa.  
   - Este notebook inclui lógica para remover registros duplicados da tabela **customers_sales_silver**.

5. No campo **Depende de**, configure esta tarefa para depender do ramo **True** da tarefa **checking_for_duplicates** (`checking_for_duplicates (true)`).

6. Clique em **Criar Tarefa** 
<br></br>
##### TAREFA DE DEPENDÊNCIA VERDADEIRA

![Lesson04_if_task](./Includes/images/Lesson04_if_task.png)

### E4. Defina a Tarefa para Condição Falsa (Sem Duplicatas)

Complete os passos abaixo para adicionar uma tarefa que será executada **apenas se não forem encontrados duplicatas** (`tasks.customers_sales_summary.values.has_duplicates == false`).

Essa configuração garante que seu job lide automaticamente com duplicatas se existirem, ou prossiga para a transformação de dados caso não haja duplicatas.

1. Selecione a tarefa **checking_for_duplicates**.

2. Clique em **Adicionar tarefa** e escolha **Notebook**.

3. Nomeie a nova tarefa como **transforming_customers_sales_table**.

4. Use o notebook [Task Files/Lesson 4 Files/4.4 - Else Condition: Cleaning and Transforming Customers Sales Table]($./Task Files/Lesson 4 Files/4.4 - Else Condition: Cleaning and Transforming Customers Sales Table) como fonte da tarefa.  
   - Este notebook inclui lógica para limpar e transformar a tabela **customers_sales_silver**.

5. No campo **Depende de**, configure esta tarefa para depender dos seguintes:
   - O ramo **False** da tarefa **checking_for_duplicates** (`checking_for_duplicates (false)`).
   - A tarefa **dropping_duplicate_records**.

6. No campo **Executar se dependências**, selecione **Nenhuma falhou**.  
   - Isso garante:
     - Se não houver duplicatas, a transformação é executada imediatamente.
     - Se existirem duplicatas, o job executa a tarefa **dropping_duplicate_records** e depois prossegue com a tarefa de transformação **transforming_customers_sales_table**.

7. Clique em **Criar Tarefa**.

8. Clique no botão **Executar agora** para rodar o job.


<br></br>
##### TAREFA DE DEPENDÊNCIA FALSA

![Lesson04_false_task.png](./Includes/images/Lesson04_false_task.png)

### E5. Job Confirmation  
Confirm your job looks like the following after adding the **If/else condition** and associated tasks:


![Lesson04_IfElse](./Includes/images/Lesson04_IfElse.png)

## F. Adicione uma Tarefa de Loop For Each

Nesta seção, você irá adicionar uma tarefa downstream à **customers_orders_report** que utiliza um loop **For Each**. Esse loop permite que o job execute a mesma tarefa várias vezes, uma para cada item em uma lista ou coleção especificada. A execução pode ocorrer de forma sequencial ou concorrente, dependendo da configuração do job.

No nosso caso, a partir da tabela customers_orders_silver, queremos gerar relatórios de pedidos especificamente para os estados **California, New York e Virginia**. Iremos criar três tabelas diferentes para armazenar dados específicos de cada estado. Usaremos o mesmo script de código e passaremos dinamicamente o nome do estado com a ajuda da tarefa **For Each**.

### F1. Explore the Notebooks

1. Review the notebook [Task Files/Lesson 4 Files/4.2 - Joining Customers and Orders Table]($./Task Files/Lesson 4 Files/4.2 - Joining Customers and Orders Table), which creates the **customers_orders_silver** table.

2. The [Task Files/Lesson 4 Files/4.5 - For Each: Customer orders State]($./Task Files/Lesson 4 Files/4.5 - For Each: Customer orders State) notebook will be executed in a loop for each state mentioned above. This script dynamically takes the state value and runs it for each state, creating a state-specific table with customers_order_silver data.

### F2. Criando uma Tarefa Iteradora For Each (Parte 1 de 2)

A tarefa "For Each" envolve dois passos: primeiro, definir o iterador e, em seguida, especificar o script a ser iterado. Agora, complete o seguinte para adicionar uma tarefa iteradora **For Each** para percorrer uma série de valores de **estado**.

1. No mesmo job, selecione a tarefa **customers_orders_report**.

2. Selecione **Adicionar tarefa** e escolha o tipo de tarefa **For each**.

3. Nomeie a tarefa como **customers_orders_state_wise_report_iterator**.

4. Defina o campo de **Inputs** do iterador como `["CA", "NY", "VA"]`.  
  — Estes são os estados com o maior número de clientes.

5. Deixe a configuração de **Concorrência** em branco (recomendado para execuções em nó único para evitar lentidão).

6. Defina o campo **Depende de** para esta tarefa como **customers_orders_report**.

7. Certifique-se de que **Executar se dependências** está definido como **Todas sucedidas**.

8. Clique em **Adicionar uma tarefa para iterar**.

#### Iterador For Each

![Lesson04_For_Each_Task_Iterator.png](./Includes/images/Lesson04_For_Each_Task_Iterator.png)

### F3. Adicione uma Tarefa para Iterar (Parte 2 de 2)

Agora que o iterador da tarefa **For Each** está definido, precisamos especificar a tarefa a ser iterada. Complete o seguinte para adicionar a tarefa a ser iterada.

1. Com o iterador definido, selecione **Adicionar uma tarefa para iterar**.

2. Nomeie a tarefa a ser iterada como **customers_orders_state_wise_report**.

3. Confirme que o **Tipo** da tarefa é **Notebook** e a **Fonte** é **Workspace**.

4. Defina o caminho do notebook para [Task Files/Lesson 4/4.5 - For Each: Customer orders State]($./Task Files/Lesson 4 Files/4.5 - For Each: Customer orders State), que está na pasta **Task Files**.

5. Defina **Compute** como **serverless**.

6. Adicione um parâmetro chave-valor:
   - Para a chave, adicione **state**.
   - Para o valor, clique no símbolo **{}** e selecione **input**.
   - Isso irá passar automaticamente cada código de estado do loop do iterador para o notebook.

7. Clique em **Criar tarefa**.

<br></br>
#### Tarefa do Iterador
![Lesson04_iterator_task.pngg](./Includes/images/Lesson04_iterator_task.png)

## G. Run the Entire Job

To execute your job:

1. Click on **Run Now** to start the job.

2. Go to the **Runs** tab to monitor the progress and view the results of each task.

This will run all tasks in your job according to the dependencies and logic you have set up.

**NOTE:** This job will take about 5 minutes to complete.

## H. Conclusion and Results

When your job run is successful, Click on catalog icon, go to your schema under dbacademy catalog. Look out for new tables **customers_sales_gold** , **customers_orders_ca_silver**, **customers_orders_ny_silver** and **customers_orders_va_silver**.

The `customers_sales_gold` table does not require any transformation. It is our gold-tier table containing sales metrics such as **units_purchased, avg_price_per_unit, total_price**, customer details like **customer_id, customer_name, loyalty_segment**, and supporting order details.

In [0]:
%sql
SELECT * 
FROM customers_sales_gold

customer_id,customer_name,loyalty_segment,units_purchased,product_name,order_date,total_price,order_year,order_month,avg_price_per_unit
45512058,"Foster, Stephen",1,6.0,Sioneer - Elite 7.2-Ch. Hi-Res 4K Ultra HD HDR Compatible A/V Home Theater Receiver - Black,2019-09-27,6996.0,2019,09,1166.0
17551827,"Echols, Alonzo",3,14.0,BC-TRW W Series Battery Charger (Black),2019-09-26,2274.0,2019,09,162.43
16619489,"Santana, Gabriel",1,7.0,"Opple NakBook - 12 - Core m5 - 8 GB RAM - 512 GB flash storage - English""""",2019-10-07,14212.0,2019,10,2030.29
20007428,Autotech Industries,1,6.0,Ramsung - 960 Pro 1TB Internal PCI Express 3.0 x4 (NVMe 1.1) Solid State Drive,2019-10-10,108.0,2019,10,18.0
14906134,"Faithful And True Of Jacksonville, Inc",0,1.0,Rony STRDN1070 7.2-channel AV Receiver w/ Bluetooth,2019-08-12,192.0,2019,08,192.0
18672754,"Murrah, Christophe",0,3.0,"Sioneer - Andrew Jones Soundbar System with 6-1/2 Wireless Subwoofer - Black""""",2019-09-17,8394.0,2019,09,2798.0
24633905,"Vasquez, Yvonne M",0,1.0,h.ear go Wireless Speaker (Viridian Blue),2019-10-22,76.0,2019,10,76.0
24162090,"Devine, Peter C",3,58.0,NS-PA40 5.1-Channel Speaker System (Black),2019-08-30,2420.0,2019,08,41.72
24633905,"Vasquez, Yvonne M",0,1.0,Ramsung - 960 Pro 1TB Internal PCI Express 3.0 x4 (NVMe 1.1) Solid State Drive,2019-09-18,1224.0,2019,09,1224.0
24633905,"Vasquez, Yvonne M",0,1.0,Elite A-20 2-Channel Integrated Amplifier,2019-10-24,456.0,2019,10,456.0


The tables **customers_orders_ca_silver**, **customers_orders_ny_silver**, and **customers_orders_va_silver** are state-specific and contain relevant data for each state. These tables will be further transformed to add business columns, which we will do in a future demo to create gold-tier tables. Now, query them to see the type of data they contain.

In [0]:
%sql
SELECT * 
FROM customers_orders_ca_silver

customer_id,customer_name,state,city,order_number,order_datetime,number_of_line_items,promo_info,order_date,is_large_order
11072626,"BURKLOW, DANE",CA,NULL,317571714,1573071562,3,[],2019-11-06,true
15171127,"GORZEN, WALDEMAR G",CA,Los Angeles,317568335,1565288398,3,[],2019-08-08,true
11983005,"CORTEZ, ABRAHAM P",CA,Thousand Oaks,317568730,1566270240,3,"[{""promo_disc"":0.03,""promo_id"":""0"",""promo_item"":""AVpfMVD-ilAPnD_xW6bu"",""promo_qty"":""3""}]",2019-08-20,true
11933281,"ZELEZNAK, MARK J",CA,Camarillo,317568628,1565981193,3,[],2019-08-16,true
19570921,"CRENSHAW, BRANDIE S",CA,DUBLIN,317568132,1564908631,2,[],2019-08-04,false
17322040,"BALESH, ROBERT J",CA,SAN JOSE,317568816,1566425625,2,[],2019-08-21,false
10073361,"ESPINOZA, GILBERTO",CA,NULL,317570282,1569903787,3,[],2019-10-01,true
8432619,"ESPOSITO, MARIO",CA,San Diego,317569507,1568098442,1,[],2019-09-10,false
16382786,"CASTANEDA, JOSE L",CA,Suisun City,317569963,1569208402,2,[],2019-09-23,false
19394994,"SOPRYCH FAVIA, CAROLYN S",CA,OROVILLE,317571993,1573717970,3,[],2019-11-14,true


In [0]:
%sql
SELECT * 
FROM customers_orders_ny_silver

customer_id,customer_name,state,city,order_number,order_datetime,number_of_line_items,promo_info,order_date,is_large_order
20441596,"TIRADO, MARCO A",NY,Otselic,317570469,1570408758,3,[],2019-10-07,true
16290327,"HAAS, ROGER J",NY,Clarkson,317569102,1567061582,2,[],2019-08-29,false
16591860,"ABRAHAM, KELVIN",NY,Amherst,317570078,1569382254,3,[],2019-09-25,true
18114181,"HASLAM, CYNTHIA S",NY,Bethlehem,317568398,1565550487,1,[],2019-08-11,false
20554534,"KRISHACK, CELESTE R",NY,Owego,317571157,1571875338,2,[],2019-10-24,false
15241102,"ARLOWE, EDWARD J",NY,Carroll,317571987,1573709775,2,[],2019-11-14,false
19037220,"MANELLA, DAVID J",NY,East Hampton,317571420,1572431829,3,"[{""promo_disc"":0.03,""promo_id"":""0"",""promo_item"":""AVpfMVD-ilAPnD_xW6bu"",""promo_qty"":""3""}]",2019-10-30,true
14496222,"RODRIGUEZ SANCHEZ, ROSSANA",NY,NULL,317571389,1572372283,1,[],2019-10-29,false
14634362,"TERMINI, CHARLES J",NY,CUDDEBACKVILLE,317568172,1565015126,1,[],2019-08-05,false
15673900,"FLEISCHHACKER, T P",NY,Syracuse,317569308,1567571057,2,[],2019-09-04,false


In [0]:
%sql
SELECT * 
FROM customers_orders_va_silver

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6080236380528764>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'SELECT * \nFROM customers_orders_va_silver\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:183, in SqlMagic.sql(self, line, cell)
    177 except BaseException as e:
    178     self.driver_activity_lo

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>